In [25]:
!pip install torch torchvision transformers sentence-transformers faiss-cpu datasets -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 31.3 MB/s eta 0:00:00


In [45]:
import torch
import random
import transformers
from torch.utils.data import Dataset, DataLoader
from transformers import BertConfig, BertModel, AutoTokenizer, get_linear_schedule_with_warmup
from sentence_transformers import SentenceTransformer, models, losses
transformers.AdamW = torch.optim.AdamW
from tqdm import tqdm
import json, os
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
import math

In [13]:
from datasets import load_dataset

dataset = load_dataset("w95/triplets")
print(dataset)

Loading dataset shards:   0%|          | 0/40 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['query', 'pos', 'neg'],
        num_rows: 3124572
    })
})


In [21]:
# Print a few examples from the dataset
print("Examples from the dataset:")
for i in range(3):
    print(f"Example {i+1}:")
    print(f"  Query: {dataset['train'][i]['query']}")
    print(f"  Positive: {dataset['train'][i]['pos']}")
    print(f"  Negative: {dataset['train'][i]['neg']}")

Examples from the dataset:
Example 1:
  Query: A person on a horse jumps over a broken down airplane.
  Positive: ['A person is outdoors, on a horse.']
  Negative: ['A white horse is jumping.', 'Someone jumps a horse over a gate.', 'A person is riding a animal.', 'The person is riding a horse', 'A horse jumps.', 'A rider rides a horse.', 'A person is jumping off a plane.', 'A horse jumping a hurdle.', 'The person is flying a plane.', 'Someone is on a horse.', 'A horse rider is jumping over a fence.', 'The rider is on a horse.', 'Someone is in a stable with a horse.', 'There is someone on top of a horse.', 'A person is flying through the air.']
Example 2:
  Query: Children smiling and waving at camera
  Positive: ['There are children present']
  Negative: ['Children are posing for the camera.', 'The children are kids.', 'Children pose for the camera', 'Children are taking a picture.', 'Children are being playful in front of the camera.', 'children are together', 'A group of people are w

In [46]:
class SmallEmbeddingModel(nn.Module):
    def __init__(self, vocab_size, d_model=384, nhead=6, num_layers=6,
                 dim_feedforward=1536, max_seq_length=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.max_seq_length = max_seq_length

        # Token and position embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_length, d_model)

        # Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Layer normalization
        self.layer_norm = nn.LayerNorm(d_model)

        # Dropout
        self.dropout = nn.Dropout(dropout)

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        # Initialize embeddings
        nn.init.normal_(self.token_embedding.weight, std=0.02)
        nn.init.normal_(self.position_embedding.weight, std=0.02)

    def mean_pooling(self, token_embeddings, attention_mask):
        # Expand attention mask
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        # Sum embeddings and divide by number of tokens
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask):
        # Get embeddings
        batch_size, seq_length = input_ids.size()

        # Token embeddings
        token_embeds = self.token_embedding(input_ids)

        # Position embeddings
        positions = torch.arange(seq_length, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        position_embeds = self.position_embedding(positions)

        # Combine embeddings
        embeddings = token_embeds + position_embeds
        embeddings = self.dropout(self.layer_norm(embeddings))

        # Create padding mask for transformer (True for padding tokens)
        padding_mask = (attention_mask == 0)

        # Pass through transformer
        encoded = self.transformer_encoder(embeddings, src_key_padding_mask=padding_mask)

        # Mean pooling
        sentence_embeddings = self.mean_pooling(encoded, attention_mask)

        # Normalize embeddings
        sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

        return sentence_embeddings



In [53]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TripletLoss(nn.Module):
    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings_tuple):
        # Unpack the embeddings from the tuple
        anchor, positive, negative = embeddings_tuple

        # Cosine similarity
        pos_sim = F.cosine_similarity(anchor, positive)
        neg_sim = F.cosine_similarity(anchor, negative)

        # Triplet loss: max(0, margin - (pos_sim - neg_sim))
        loss = torch.clamp(self.margin - (pos_sim - neg_sim), min=0.0)
        return loss.mean()

In [51]:
def collate_fn(batch, tokenizer, max_length=128):
    queries = [item['query'] for item in batch]
    positives = [item['pos'][0] for item in batch]  # Take first positive
    negatives = [item['neg'][0] for item in batch]  # Take first negative

    # Tokenize
    query_encoded = tokenizer(queries, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    pos_encoded = tokenizer(positives, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    neg_encoded = tokenizer(negatives, padding=True, truncation=True, max_length=max_length, return_tensors='pt')

    return {
        'query': query_encoded,
        'positive': pos_encoded,
        'negative': neg_encoded
    }

In [55]:
def train_model(model, train_loader, optimizer, criterion, device, num_epochs=3):
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')

        for batch_idx, batch in enumerate(progress_bar):
            # Move to device
            query_ids = batch['query']['input_ids'].to(device)
            query_mask = batch['query']['attention_mask'].to(device)
            pos_ids = batch['positive']['input_ids'].to(device)
            pos_mask = batch['positive']['attention_mask'].to(device)
            neg_ids = batch['negative']['input_ids'].to(device)
            neg_mask = batch['negative']['attention_mask'].to(device)

            # Forward pass
            query_embeds = model(query_ids, query_mask)
            pos_embeds = model(pos_ids, pos_mask)
            neg_embeds = model(neg_ids, neg_mask)

            # Calculate loss - Pass embeddings as a tuple
            loss = criterion((query_embeds, pos_embeds, neg_embeds))

            # Backward pass
            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            total_loss += loss.item()

            # Update progress bar
            progress_bar.set_postfix({'loss': loss.item()})

        avg_loss = total_loss / len(train_loader)
        print(f'Epoch {epoch+1} - Average Loss: {avg_loss:.4f}')

In [56]:
def main():
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')

    # Load tokenizer (using pretrained tokenizer)
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

    # Load dataset
    print('Loading dataset...')
    # Correct dataset loading to w95/triplets
    dataset = load_dataset('w95/triplets')

    # Take a subset for faster training (remove this line to use full dataset)
    train_dataset = dataset['train'].select(range(10000))  # Use 10k samples

    # Create model from scratch
    print('Creating model...')
    model = SmallEmbeddingModel(
        vocab_size=tokenizer.vocab_size,
        d_model=384,           # Embedding dimension
        nhead=6,               # Number of attention heads
        num_layers=6,          # Number of transformer layers
        dim_feedforward=1536,  # FFN dimension
        max_seq_length=128,
        dropout=0.1
    )
    model.to(device)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total parameters: {total_params:,}')
    print(f'Trainable parameters: {trainable_params:,}')

    # Create data loader
    # Ensure collate_fn is correctly defined and accessible
    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,
        collate_fn=lambda x: collate_fn(x, tokenizer), # Use the collate_fn defined in another cell
        num_workers=2
    )

    # Loss and optimizer
    criterion = TripletLoss(margin=0.5) # Ensure TripletLoss is defined and accessible
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

    # Training
    print('Starting training...')
    # Ensure train_model is defined and accessible
    train_model(model, train_loader, optimizer, criterion, device, num_epochs=3)

    # Save model
    print('Saving model...')
    torch.save({
        'model_state_dict': model.state_dict(),
        'model_config': {
            'vocab_size': tokenizer.vocab_size,
            'd_model': 384,
            'nhead': 6,
            'num_layers': 6,
            'dim_feedforward': 1536,
            'max_seq_length': 128,
            'dropout': 0.1
        }
    }, 'small_embedding_model.pt')
    print('Model saved!')

    # Test the model
    print('\nTesting model...')
    model.eval()
    test_sentences = [
        "A person on a horse jumps over a broken down airplane.",
        "A person is outdoors, on a horse.",
        "The cat is sleeping on the couch."
    ]

    with torch.no_grad():
        encoded = tokenizer(test_sentences, padding=True, truncation=True,
                          max_length=128, return_tensors='pt')
        encoded = {k: v.to(device) for k, v in encoded.items()}
        # Need to pass attention_mask separately for this model's forward
        embeddings = model(encoded['input_ids'], encoded['attention_mask'])

        print('\nSimilarity scores:')
        for i in range(len(test_sentences)):
            for j in range(i+1, len(test_sentences)):
                sim = F.cosine_similarity(embeddings[i:i+1], embeddings[j:j+1])
                print(f'Sentence {i+1} vs {j+1}: {sim.item():.4f}')


if __name__ == '__main__':
    main()

Using device: cuda
Loading dataset...


Loading dataset shards:   0%|          | 0/40 [00:00<?, ?it/s]

Creating model...
Total parameters: 22,417,152
Trainable parameters: 22,417,152
Starting training...


Epoch 1/3: 100%|██████████| 313/313 [00:28<00:00, 10.95it/s, loss=0.41]


Epoch 1 - Average Loss: 0.4273


Epoch 2/3: 100%|██████████| 313/313 [00:28<00:00, 11.05it/s, loss=0.352]


Epoch 2 - Average Loss: 0.3853


Epoch 3/3: 100%|██████████| 313/313 [00:28<00:00, 11.05it/s, loss=0.402]


Epoch 3 - Average Loss: 0.3512
Saving model...
Model saved!

Testing model...

Similarity scores:
Sentence 1 vs 2: 0.7634
Sentence 1 vs 3: 0.1992
Sentence 2 vs 3: 0.1837
